In [2]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn
!apt-get -qq update
!apt-get -qq install -y unrar


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# XLM-R + x-transformers — Original SIED-Thai — Strong Optimized

เวอร์ชันนี้เปลี่ยนเฉพาะ encoder เป็น `xlm-roberta-base` และใช้ **SIED-Thai ตัวต้นฉบับที่ยังไม่ balance**

- ใช้ `SIED-Thai.csv` จาก archive ที่ผู้ใช้อัปโหลด
- 2,400 แถว
- Labels: Happiness, Sadness, Anger, Disgust, Surprise, Fear
- split 70/15/15 ด้วย `SEED=42` และ stratify ตาม label combination
- คง `MAX_LEN=128`, `EPOCHS=100`, `BATCH_SIZE=64`
- คง Optuna Strong Optimized framework เดิม: `N_TRIALS=25`, `OPTUNA_EPOCHS=35`
- encoder frozen
- decoder, pooling, classifier, loss search, threshold tuning และ objective คงเดิม


In [3]:
import os
import json
import random
import warnings
import subprocess
import shutil
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss,
    multilabel_confusion_matrix
)

import optuna


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

DEVICE: cuda


In [6]:
MODEL_NAME = "xlm-roberta-base"

MAX_LEN = 128
EPOCHS = 100
BATCH_SIZE = 64
NUM_LABELS = 6

N_TRIALS = 25
OPTUNA_EPOCHS = 35

LABEL_COLUMNS = [
    "Happiness",
    "Sadness",
    "Anger",
    "Disgust",
    "Surprise",
    "Fear"
]

TEXT_COLUMN = "Tweets"


In [7]:
# ถ้าใช้ไฟล์ .rar ที่อัปโหลดมาโดยตรงใน Google Colab
RAR_PATH = "/content/SIED-Thai.rar"
CSV_PATH = "/content/SIED-Thai.csv"
EXTRACT_ROOT = "/content/sied_thai_extracted"

if not os.path.exists(CSV_PATH) and os.path.exists(RAR_PATH):
    os.makedirs(EXTRACT_ROOT, exist_ok=True)

    if shutil.which("unrar") is None:
        raise RuntimeError(
            "ไม่พบคำสั่ง unrar กรุณารัน cell ติดตั้งด้านบนก่อน "
            "หรือ extract SIED-Thai.rar แล้วอัปโหลด SIED-Thai.csv"
        )

    subprocess.run(
        ["unrar", "x", "-o+", RAR_PATH, EXTRACT_ROOT + "/"],
        check=True
    )

    CSV_PATH = EXTRACT_ROOT

# fallback สำหรับ local/sandbox
if not os.path.exists(CSV_PATH):
    kaggle_root = Path("/kaggle/input/datasets/kmkimmy/sied-thai")
    CSV_PATH = str(kaggle_root) if kaggle_root.is_dir() else "/mnt/data/SIED-Thai.csv"

# Resolve a directory to a CSV file, including nested archive folders.
csv_path = Path(CSV_PATH)
if csv_path.is_dir():
    csv_files = sorted(path for path in csv_path.rglob("*.csv") if path.is_file())
    preferred = [path for path in csv_files if path.name.lower() == "sied-thai.csv"]
    csv_files = preferred or csv_files
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in: {csv_path}")
    if len(csv_files) > 1:
        raise ValueError(f"Multiple CSV files found; set CSV_PATH to the intended file: {csv_files}")
    CSV_PATH = str(csv_files[0])

if not os.path.isfile(CSV_PATH):
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")

print("SIED CSV:", CSV_PATH)


df = pd.read_csv(CSV_PATH)

required_columns = [TEXT_COLUMN] + LABEL_COLUMNS

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"dataset ไม่มี column: {col}")

df = df[required_columns].copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

for col in LABEL_COLUMNS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).fillna(0).astype(int)

print("TOTAL ROWS:", len(df))
print("\nORIGINAL LABEL DISTRIBUTION")
print(df[LABEL_COLUMNS].sum())

print("\nORIGINAL LABEL CARDINALITY")
print(df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())


SIED CSV: /kaggle/input/datasets/kmkimmy/sied-thai/SIED-Thai.csv
TOTAL ROWS: 2400

ORIGINAL LABEL DISTRIBUTION
Happiness     170
Sadness      1886
Anger         312
Disgust       467
Surprise      162
Fear          304
dtype: int64

ORIGINAL LABEL CARDINALITY
0     112
1    1460
2     659
3     153
4      16
Name: count, dtype: int64


In [8]:
# Original SIED-Thai ไม่มี train/validation/test แยกมาให้
# ใช้ split 70/15/15 เหมือน Strong Optimized framework
# และใช้ stratify ตาม label combination เพื่อลดความคลาดเคลื่อนของ class distribution

def make_stratify_key(frame):
    label_part = frame[LABEL_COLUMNS].astype(str).agg("".join, axis=1)
    card_part = frame[LABEL_COLUMNS].sum(axis=1).astype(str)
    key = card_part + "_" + label_part

    counts = key.value_counts()
    key = key.where(key.map(counts) >= 2, "rare")

    return key


stratify_key = make_stratify_key(df)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=stratify_key
)

temp_stratify_key = make_stratify_key(temp_df)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=temp_stratify_key
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("TRAIN:", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST:", len(test_df))

print("\nTRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVAL LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nTRAIN LABEL CARDINALITY")
print(train_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())

print("\nVAL LABEL CARDINALITY")
print(val_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())

print("\nTEST LABEL CARDINALITY")
print(test_df[LABEL_COLUMNS].sum(axis=1).value_counts().sort_index())


TRAIN: 1680
VALIDATION: 360
TEST: 360

TRAIN LABEL DISTRIBUTION
Happiness     119
Sadness      1320
Anger         218
Disgust       328
Surprise      112
Fear          212
dtype: int64

VAL LABEL DISTRIBUTION
Happiness     25
Sadness      282
Anger         48
Disgust       67
Surprise      24
Fear          48
dtype: int64

TEST LABEL DISTRIBUTION
Happiness     26
Sadness      284
Anger         46
Disgust       72
Surprise      26
Fear          44
dtype: int64

TRAIN LABEL CARDINALITY
0      78
1    1023
2     461
3     108
4      10
Name: count, dtype: int64

VAL LABEL CARDINALITY
0     17
1    220
2     98
3     22
4      3
Name: count, dtype: int64

TEST LABEL CARDINALITY
0     17
1    217
2    100
3     23
4      3
Name: count, dtype: int64


In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [10]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

train_eval_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [12]:
# pos_weight เดิมทำให้ recall สูง แต่ precision ตกได้
# จึงไม่ fix pos_weight ตายตัว แต่ให้ Optuna tune pos_weight_scale
# scale = 0.0 คือไม่ใช้ pos_weight
# scale = 1.0 คือใช้ pos_weight เต็มสูตร negative / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)
total_samples = len(train_df)
neg_counts = total_samples - label_counts

pos_weights_raw = neg_counts / (label_counts + 1e-6)

pos_weights_raw = torch.tensor(
    pos_weights_raw,
    dtype=torch.float
).to(device)

def build_pos_weight(pos_weight_scale):
    pos_weight_scale = float(pos_weight_scale)

    weights = 1.0 + (pos_weights_raw - 1.0) * pos_weight_scale

    return weights.to(device)

print("LABEL COUNTS:", label_counts)
print("RAW POS WEIGHTS:", pos_weights_raw)
print("NO POS WEIGHT:", build_pos_weight(0.0))
print("HALF POS WEIGHT:", build_pos_weight(0.5))


LABEL COUNTS: [ 119. 1320.  218.  328.  112.  212.]
RAW POS WEIGHTS: tensor([13.1176,  0.2727,  6.7064,  4.1220, 14.0000,  6.9245], device='cuda:0')
NO POS WEIGHT: tensor([1., 1., 1., 1., 1., 1.], device='cuda:0')
HALF POS WEIGHT: tensor([7.0588, 0.6364, 3.8532, 2.5610, 7.5000, 3.9623], device='cuda:0')


In [13]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [14]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.15,
        ff_dropout=0.15,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # Fixed Thai encoder: train เฉพาะ decoder + pooling + classifier
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.30)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


In [15]:
def optimize_thresholds(
    y_true,
    y_probs,
    min_thr=0.20,
    max_thr=0.80,
    step=0.005
):

    thresholds = np.arange(
        min_thr,
        max_thr + 1e-9,
        step
    )

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_score = -1
        best_precision = -1

        true_support = y_true[:, i].sum()

        for thr in thresholds:

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            precision = precision_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            # ถ้า F1 เท่ากัน ให้เลือก threshold ที่ precision สูงกว่า
            # ช่วยแก้ปัญหา optimized เดิมที่ recall สูงแต่ precision ต่ำ
            if (score > best_score) or (
                np.isclose(score, best_score) and precision > best_precision
            ):

                best_score = score
                best_precision = precision
                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)


In [16]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    if thresholds is None:
        thresholds = np.array([0.5] * NUM_LABELS)

    preds = (all_probs >= thresholds).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    per_label_f1 = f1_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_precision = precision_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    per_label_recall = recall_score(
        all_labels,
        preds,
        average=None,
        zero_division=0
    )

    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "per_label_f1": per_label_f1,
        "per_label_precision": per_label_precision,
        "per_label_recall": per_label_recall,
        "min_per_label_f1": float(np.min(per_label_f1)),
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


def objective_score(result):
    # objective ใหม่:
    # - macro_f1 สำคัญที่สุด เพราะต้องชนะราย class
    # - micro_f1 ยังสำคัญ
    # - min_per_label_f1 กันไม่ให้ class ใด class หนึ่งต่ำ โดยเฉพาะ Disgust
    # - subset accuracy ช่วยให้ทายชุด label ตรงขึ้น
    return (
        0.45 * result["macro_f1"]
        + 0.25 * result["micro_f1"]
        + 0.20 * result["min_per_label_f1"]
        + 0.10 * result["subset_accuracy"]
    )


In [17]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay,
    epochs=None,
    pos_weight_scale=0.0,
    label_smoothing=0.0,
    loss_name="bce",
    focal_gamma=2.0,
    verbose=True
):

    if epochs is None:
        epochs = EPOCHS

    pos_weight = build_pos_weight(pos_weight_scale)

    bce_loss = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight,
        reduction="none"
    )

    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * epochs

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        loop = tqdm(
            train_loader,
            disable=not verbose
        )

        for batch in loop:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            if label_smoothing > 0:
                # smooth toward 0.5 for multi-label BCE
                labels_for_loss = labels * (1.0 - label_smoothing) + 0.5 * label_smoothing
            else:
                labels_for_loss = labels

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss_matrix = bce_loss(
                    logits,
                    labels_for_loss
                )

                if loss_name == "focal":
                    probs = torch.sigmoid(logits)
                    pt = probs * labels + (1 - probs) * (1 - labels)
                    focal_weight = (1 - pt).pow(focal_gamma)
                    loss = (focal_weight * loss_matrix).mean()
                else:
                    loss = loss_matrix.mean()

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

            if verbose:
                loop.set_description(
                    f"Epoch {epoch+1}/{epochs}"
                )

                loop.set_postfix(
                    loss=loss.item()
                )

        if verbose:

            val_raw = evaluate(
                model,
                val_loader
            )

            val_thresholds = optimize_thresholds(
                val_raw["labels"],
                val_raw["probs"]
            )

            val_result = evaluate(
                model,
                val_loader,
                val_thresholds
            )

            print("\n======================")
            print(f"Epoch {epoch+1}")
            print("======================")

            print(
                f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
            )

            print(
                f"VAL MICRO P/R/F1: "
                f"{val_result['micro_precision']:.4f} / "
                f"{val_result['micro_recall']:.4f} / "
                f"{val_result['micro_f1']:.4f}"
            )

            print(
                f"VAL MACRO P/R/F1: "
                f"{val_result['macro_precision']:.4f} / "
                f"{val_result['macro_recall']:.4f} / "
                f"{val_result['macro_f1']:.4f}"
            )

            print(
                f"VAL MIN LABEL F1: {val_result['min_per_label_f1']:.4f}"
            )

            print(
                f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
            )

            print(
                f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
            )

    return model


In [18]:
def objective(trial):

    # baseline trial จะถูก enqueue ไว้ก่อนหน้า
    depth = trial.suggest_int(
        "depth",
        1,
        3
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4, 8]
    )

    # ลด dropout range จากเดิมที่สูงเกินไป เพราะเดิม optimized มีแนวโน้ม underfit / recall เยอะ precision ตก
    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.05,
        0.30
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.05,
        0.30
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        1e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        5e-3,
        log=True
    )

    # จุดสำคัญ: ให้ Optuna เลือกว่าจะใช้ pos_weight แค่ไหน
    # ถ้า scale สูงเกินจะ recall สูงแต่ precision ลด
    pos_weight_scale = trial.suggest_float(
        "pos_weight_scale",
        0.0,
        0.60
    )

    label_smoothing = trial.suggest_float(
        "label_smoothing",
        0.0,
        0.04
    )

    loss_name = trial.suggest_categorical(
        "loss_name",
        ["bce", "focal"]
    )

    focal_gamma = trial.suggest_float(
        "focal_gamma",
        1.0,
        2.5
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr=lr,
        weight_decay=weight_decay,
        epochs=OPTUNA_EPOCHS,
        pos_weight_scale=pos_weight_scale,
        label_smoothing=label_smoothing,
        loss_name=loss_name,
        focal_gamma=focal_gamma,
        verbose=False
    )

    val_raw = evaluate(
        model,
        val_loader
    )

    thresholds = optimize_thresholds(
        val_raw["labels"],
        val_raw["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        thresholds
    )

    score = objective_score(val_result)

    trial.set_user_attr("val_macro_f1", float(val_result["macro_f1"]))
    trial.set_user_attr("val_micro_f1", float(val_result["micro_f1"]))
    trial.set_user_attr("val_min_per_label_f1", float(val_result["min_per_label_f1"]))
    trial.set_user_attr("val_subset_accuracy", float(val_result["subset_accuracy"]))
    trial.set_user_attr("thresholds", thresholds.tolist())

    return score


In [19]:
sampler = optuna.samplers.TPESampler(
    seed=SEED,
    multivariate=True
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# ใส่ค่า baseline/non-optimized configuration เป็น trial แรก
# เพื่อให้ optimization มีจุดเริ่มต้นที่ไม่แพ้ baseline ง่าย ๆ
study.enqueue_trial({
    "depth": 1,
    "heads": 2,
    "attn_dropout": 0.20,
    "ff_dropout": 0.20,
    "ff_mult": 2,
    "lr": 2e-5,
    "weight_decay": 1e-2,
    "pos_weight_scale": 0.0,
    "label_smoothing": 0.0,
    "loss_name": "bce",
    "focal_gamma": 2.0
})

study.optimize(
    objective,
    n_trials=N_TRIALS
)


[I 2026-09-26 06:05:57,039] A new study created in memory with name: no-name-d87ce556-42b7-47ea-94a7-53aac0b85124


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-09-26 06:08:25,856] Trial 0 finished with value: 0.2890017194792434 and parameters: {'depth': 1, 'heads': 2, 'attn_dropout': 0.2, 'ff_dropout': 0.2, 'ff_mult': 2, 'lr': 2e-05, 'weight_decay': 0.01, 'pos_weight_scale': 0.0, 'label_smoothing': 0.0, 'loss_name': 'bce', 'focal_gamma': 2.0}. Best is trial 0 with value: 0.2890017194792434.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-09-26 06:11:08,492] Trial 1 finished with value: 0.4676831304678729 and parameters: {'depth': 2, 'heads': 2, 'attn_dropout': 0.08900466011060913, 'ff_dropout': 0.08899863008405066, 'ff_mult': 2, 'lr': 7.348118405270451e-05, 'weight_decay': 0.00016730402817820238, 'pos_weight_scale': 0.4248435466776273, 'label_smoothing': 0.0008233797718320979, 'loss_name': 'bce', 'focal_gamma': 1.3185086660174141}. Best is trial 1 with value: 0.4676831304678729.


...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-09-26 07:13:10,132] Trial 22 finished with value: 0.5176440415037511 and parameters: {'depth': 3, 'heads': 2, 'attn_dropout': 0.2959113531668028, 'ff_dropout': 0.07187309687062957, 'ff_mult': 2, 'lr': 8.037413975135961e-05, 'weight_decay': 6.935833306658103e-06, 'pos_weight_scale': 0.4696935860688663, 'label_smoothing': 0.022073726610620712, 'loss_name': 'focal', 'focal_gamma': 1.1146146957308125}. Best is trial 18 with value: 0.5262226237972273.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-09-26 07:16:08,064] Trial 23 finished with value: 0.5238378532521872 and parameters: {'depth': 3, 'heads': 2, 'attn_dropout': 0.244209788179082, 'ff_dropout': 0.055175350018493656, 'ff_mult': 2, 'lr': 9.782602379171752e-05, 'weight_decay': 5.7333152052377255e-06, 'pos_weight_scale': 0.25951984636752884, 'label_smoothing': 0.023428711967139432, 'loss_name': 'focal', 'focal_gamma': 1.4162822483976738}. Best is trial 18 with value: 0.5262226237972273.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[I 2026-09-26 07:19:05,989] Trial 24 finished with value: 0.5265822767726044 and parameters: {'depth': 3, 'heads': 2, 'attn_dropout': 0.25519213354804254, 'ff_dropout': 0.08837295525787028, 'ff_mult': 2, 'lr': 9.64971213067132e-05, 'weight_decay': 1.2592622143106602e-06, 'pos_weight_scale': 0.41847972523828914, 'label_smoothing': 0.0345064968680565, 'loss_name': 'focal', 'focal_gamma': 1.5282173040307059}. Best is trial 24 with value: 0.5265822767726044.


In [20]:
print("\n======================")
print("BEST PARAMETERS")
print("======================")

print(study.best_params)

print("\n======================")
print("BEST TRIAL VALIDATION ATTRS")
print("======================")

print("objective_score:", study.best_value)
print("val_macro_f1:", study.best_trial.user_attrs.get("val_macro_f1"))
print("val_micro_f1:", study.best_trial.user_attrs.get("val_micro_f1"))
print("val_min_per_label_f1:", study.best_trial.user_attrs.get("val_min_per_label_f1"))
print("val_subset_accuracy:", study.best_trial.user_attrs.get("val_subset_accuracy"))

best_params = study.best_params



BEST PARAMETERS
{'depth': 3, 'heads': 2, 'attn_dropout': 0.25519213354804254, 'ff_dropout': 0.08837295525787028, 'ff_mult': 2, 'lr': 9.64971213067132e-05, 'weight_decay': 1.2592622143106602e-06, 'pos_weight_scale': 0.41847972523828914, 'label_smoothing': 0.0345064968680565, 'loss_name': 'focal', 'focal_gamma': 1.5282173040307059}

BEST TRIAL VALIDATION ATTRS
objective_score: 0.5265822767726044
val_macro_f1: 0.5445434428158554
val_micro_f1: 0.6978354978354978
val_min_per_label_f1: 0.3548387096774194
val_subset_accuracy: 0.3611111111111111


In [21]:
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
    epochs=EPOCHS,
    pos_weight_scale=best_params["pos_weight_scale"],
    label_smoothing=best_params["label_smoothing"],
    loss_name=best_params["loss_name"],
    focal_gamma=best_params["focal_gamma"],
    verbose=True
)


Epoch 1/1: 100%|██████████| 27/27 [00:04<00:00,  5.57it/s, loss=0.383]



Epoch 1
VAL SUBSET ACC : 0.0000
VAL MICRO P/R/F1: 0.2764 / 0.9474 / 0.4280
VAL MACRO P/R/F1: 0.2506 / 0.8749 / 0.3461
VAL MIN LABEL F1: 0.1304
VAL WEIGHTED F1: 0.6136
VAL HAMMING LOSS: 0.5792


In [23]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

val_result_tuned = evaluate(
    final_model,
    val_loader,
    best_thresholds
)

print("\n======================")
print("BEST THRESHOLDS FROM FINAL VALIDATION")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.3f}")

print("\nVALIDATION RESULT AFTER THRESHOLD TUNING")
print("macro_f1:", val_result_tuned["macro_f1"])
print("micro_f1:", val_result_tuned["micro_f1"])
print("min_per_label_f1:", val_result_tuned["min_per_label_f1"])
print("subset_accuracy:", val_result_tuned["subset_accuracy"])



BEST THRESHOLDS FROM FINAL VALIDATION
Happiness: 0.440
Sadness: 0.585
Anger: 0.335
Disgust: 0.410
Surprise: 0.375
Fear: 0.375

VALIDATION RESULT AFTER THRESHOLD TUNING
macro_f1: 0.3461327587077212
micro_f1: 0.4279835390946502
min_per_label_f1: 0.13043478260869565
subset_accuracy: 0.0


In [24]:
def summarize_result(name, result):

    return {
        "split": name,
        "subset_accuracy_exact_match": result["subset_accuracy"],
        "micro_precision": result["micro_precision"],
        "micro_recall": result["micro_recall"],
        "micro_f1": result["micro_f1"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
        "weighted_f1": result["weighted_f1"],
        "min_per_label_f1": result["min_per_label_f1"],
        "hamming_loss": result["hamming_loss"]
    }


train_result = evaluate(
    final_model,
    train_eval_loader,
    best_thresholds
)

val_result_final = evaluate(
    final_model,
    val_loader,
    best_thresholds
)

test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds
)

metrics_summary = pd.DataFrame([
    summarize_result("train", train_result),
    summarize_result("validation", val_result_final),
    summarize_result("test", test_result)
])

print("\n======================")
print("FINAL METRICS SUMMARY")
print("======================")
display(metrics_summary)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")

threshold_df = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "threshold": best_thresholds
})

display(threshold_df)

print("\n======================")
print("TEST SET MAIN RESULT")
print("======================")

print(f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}")
print(f"Micro Precision              : {test_result['micro_precision']:.4f}")
print(f"Micro Recall                 : {test_result['micro_recall']:.4f}")
print(f"Micro F1                     : {test_result['micro_f1']:.4f}")
print(f"Macro Precision              : {test_result['macro_precision']:.4f}")
print(f"Macro Recall                 : {test_result['macro_recall']:.4f}")
print(f"Macro F1                     : {test_result['macro_f1']:.4f}")
print(f"Weighted F1                  : {test_result['weighted_f1']:.4f}")
print(f"Min Per-label F1             : {test_result['min_per_label_f1']:.4f}")
print(f"Hamming Loss                 : {test_result['hamming_loss']:.4f}")



FINAL METRICS SUMMARY


,split,subset_accuracy_exact_match,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,weighted_f1,min_per_label_f1,hamming_loss
0,train,0.001786,0.270705,0.911650,0.417452,0.243486,0.828230,0.332994,0.602803,0.126377,0.582837
1,validation,0.000000,0.276432,0.947368,0.427984,0.250572,0.874942,0.346133,0.613612,0.130435,0.579167
2,test,0.000000,0.275208,0.931727,0.424908,0.255097,0.871744,0.350799,0.609425,0.138298,0.581481



BEST THRESHOLDS


,label,threshold
0,Happiness,0.440
1,Sadness,0.585
2,Anger,0.335
3,Disgust,0.410
4,Surprise,0.375
5,Fear,0.375



TEST SET MAIN RESULT
Subset Accuracy / Exact Match : 0.0000
Micro Precision              : 0.2752
Micro Recall                 : 0.9317
Micro F1                     : 0.4249
Macro Precision              : 0.2551
Macro Recall                 : 0.8717
Macro F1                     : 0.3508
Weighted F1                  : 0.6094
Min Per-label F1             : 0.1383
Hamming Loss                 : 0.5815


In [25]:
print("\n======================")
print("TEST CLASSIFICATION REPORT")
print("======================")

report_dict = classification_report(
    test_result["labels"],
    test_result["preds"],
    target_names=LABEL_COLUMNS,
    zero_division=0,
    output_dict=True
)

per_label_report = pd.DataFrame(report_dict).T
display(per_label_report)

per_label_metrics = pd.DataFrame({
    "label": LABEL_COLUMNS,
    "precision": test_result["per_label_precision"],
    "recall": test_result["per_label_recall"],
    "f1_score": test_result["per_label_f1"],
    "true_support": test_result["labels"].sum(axis=0).astype(int),
    "predicted_positive": test_result["preds"].sum(axis=0).astype(int),
    "threshold": best_thresholds
})

print("\n======================")
print("PER-LABEL TEST METRICS")
print("======================")
display(per_label_metrics)

print("\n======================")
print("TEST MULTI-LABEL CONFUSION MATRIX")
print("======================")

mcm = multilabel_confusion_matrix(
    test_result["labels"],
    test_result["preds"]
)

confusion_rows = []

for i, label in enumerate(LABEL_COLUMNS):
    tn, fp, fn, tp = mcm[i].ravel()

    confusion_rows.append({
        "label": label,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn)
    })

confusion_df = pd.DataFrame(confusion_rows)
display(confusion_df)

test_probs_df = pd.DataFrame(
    test_result["probs"],
    columns=[f"prob_{label}" for label in LABEL_COLUMNS]
)

test_preds_df = pd.DataFrame(
    test_result["preds"],
    columns=[f"pred_{label}" for label in LABEL_COLUMNS]
)

test_true_df = test_df[[TEXT_COLUMN] + LABEL_COLUMNS].copy()

test_predictions_output = pd.concat(
    [
        test_true_df.reset_index(drop=True),
        test_probs_df.reset_index(drop=True),
        test_preds_df.reset_index(drop=True)
    ],
    axis=1
)

metrics_summary.to_csv(
    "/content/xlmr_sied_original_metrics_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_df.to_csv(
    "/content/xlmr_sied_original_thresholds.csv",
    index=False,
    encoding="utf-8-sig"
)

per_label_report.to_csv(
    "/content/xlmr_sied_original_per_label_report.csv",
    encoding="utf-8-sig"
)

per_label_metrics.to_csv(
    "/content/xlmr_sied_original_per_label_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

confusion_df.to_csv(
    "/content/xlmr_sied_original_confusion_matrix.csv",
    index=False,
    encoding="utf-8-sig"
)

test_predictions_output.to_csv(
    "/content/xlmr_sied_original_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nEXPORTED FILES")
print("/content/xlmr_sied_original_metrics_summary.csv")
print("/content/xlmr_sied_original_thresholds.csv")
print("/content/xlmr_sied_original_per_label_report.csv")
print("/content/xlmr_sied_original_per_label_metrics.csv")
print("/content/xlmr_sied_original_confusion_matrix.csv")
print("/content/xlmr_sied_original_test_predictions.csv")



TEST CLASSIFICATION REPORT


,precision,recall,f1-score,support
Happiness,0.192308,0.576923,0.288462,26.0
Sadness,0.786325,0.971831,0.869291,284.0
Anger,0.133987,0.891304,0.232955,46.0
Disgust,0.204678,0.972222,0.338164,72.0
Surprise,0.074286,1.000000,0.138298,26.0
Fear,0.138996,0.818182,0.237624,44.0
micro avg,0.275208,0.931727,0.424908,498.0
macro avg,0.255097,0.871744,0.350799,498.0
weighted avg,0.516594,0.931727,0.609425,498.0
samples avg,0.272407,0.881019,0.403060,498.0



PER-LABEL TEST METRICS


,label,precision,recall,f1_score,true_support,predicted_positive,threshold
0,Happiness,0.192308,0.576923,0.288462,26,78,0.440
1,Sadness,0.786325,0.971831,0.869291,284,351,0.585
2,Anger,0.133987,0.891304,0.232955,46,306,0.335
3,Disgust,0.204678,0.972222,0.338164,72,342,0.410
4,Surprise,0.074286,1.000000,0.138298,26,350,0.375
5,Fear,0.138996,0.818182,0.237624,44,259,0.375



TEST MULTI-LABEL CONFUSION MATRIX


,label,TP,FP,FN,TN
0,Happiness,15,63,11,271
1,Sadness,276,75,8,1
2,Anger,41,265,5,49
3,Disgust,70,272,2,16
4,Surprise,26,324,0,10
5,Fear,36,223,8,93



EXPORTED FILES
/content/xlmr_sied_original_metrics_summary.csv
/content/xlmr_sied_original_thresholds.csv
/content/xlmr_sied_original_per_label_report.csv
/content/xlmr_sied_original_per_label_metrics.csv
/content/xlmr_sied_original_confusion_matrix.csv
/content/xlmr_sied_original_test_predictions.csv


In [26]:
SAVE_PATH = "/content/final_xlmr_sied_original_xtransformers_strong_optimized.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "dataset": "SIED-Thai original/unbalanced (2,400 rows)",
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optuna_trials": N_TRIALS,
        "optuna_epochs": OPTUNA_EPOCHS,
        "objective": "0.45 macro_f1 + 0.25 micro_f1 + 0.20 min_per_label_f1 + 0.10 subset_accuracy",
        "note": "Encoder comparison version; encoder frozen; same Strong Optimized EAQD/x-transformers framework."
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)



MODEL SAVED: /content/final_xlmr_sied_original_xtransformers_strong_optimized.pt


In [1]:
import shutil
from IPython.display import FileLink, display

OUTPUT_PATH = "/kaggle/working/final_xlmr_sied_original_xtransformers_strong_optimized.pt"

# คัดลอกโมเดลที่เซฟแล้ว ไม่ต้องเทรนใหม่
shutil.copy2(SAVE_PATH, OUTPUT_PATH)
display(FileLink(OUTPUT_PATH))

NameError: name 'SAVE_PATH' is not defined